[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/astudentfromsustech/agents-course/blob/main/practice/p2_vit_mnist/vit_mnist.ipynb)

# A transformer that reads handwritten digits

An image is not a sequence. A **Vision Transformer** makes it one: cut the 28x28 image into
7x7 patches, project each to a vector, and call that the sequence. Everything after that is
chapter 01 unchanged — Q/K/V, softmax, multi-head, residual, norm.

The point is not the classifier. It is that three things the chapter *claims* become
**measurable** here:

| | the claim | the measurement |
|---|---|---|
| **position** | attention cannot see order | strip the positional encoding, shuffle the patches: accuracy changes by **exactly +0.00** |
| **scale** | a CNN's locality is a free prior | the gap closes from **-19.4** to **-2.9** as data grows |
| **attention** | the weights are interpretable | the `[CLS]` map runs up the stroke of a `1`, around the ring of a `0` |

Parts 1-3 are what you need to read those three numbers. Part 4 measures them.
Whole notebook ≈ 2 minutes on a GPU.

In [ ]:
# Setup — works on Colab and locally, no torchvision needed.
import os, sys, importlib, urllib.request
URL = ("https://raw.githubusercontent.com/astudentfromsustech/agents-course"
       "/main/practice/p2_vit_mnist/vit_mnist.py")
if "google.colab" in sys.modules or not os.path.exists("vit_mnist.py"):
    urllib.request.urlretrieve(URL, "vit_mnist.py")   # on Colab always refetch
import vit_mnist as V
importlib.reload(V)                                   # a warm kernel holds the old module

from IPython.display import Image, display
d = V.load()
print("device:", V.DEV)        # mps = your Mac's GPU | cuda = Colab GPU | cpu = fallback

---
## 1 · What goes in

28x28 pixels become 16 tokens. This part is only about that step.

In [ ]:
display(Image(V.show_data(d)))

**The bottom strip is the entire input.** The model never receives the 4x4 arrangement —
only those 16 tokens in order, plus a 17th we will meet shortly. Token 5 and 6 hold the top bar
of the `7`, 10 and 13 the diagonal.

Each token is **64 numbers**, not 49: the patch is 7x7 = 49 pixels, and a learned matrix maps
`49 -> 64`. Here is the real matrix for one digit.

In [ ]:
display(Image(V.show_tokens(d)))

Rows 5, 6, 10, 13 carry strong values — the inked patches. The eight pale rows outlined in
green are the blank patches, and they are **byte-identical to each other**: a blank patch is 49
copies of the same background value, so its token is `W @ (constant) + bias` wherever it sat.

**So at this point token 0 and token 15 are indistinguishable.** Nothing about *where* a patch
sat has entered yet — which is exactly why experiment 2 comes out the way it does.

### Why 64, when the patch is 49 pixels?

**49 is forced** by the patch size. **64 is chosen** — it is `d_model`, and nothing derives it
from 49. Two questions follow: what is in the extra 15 dimensions, and is `49 -> 64` a real
transformation or just zero-padding?

In [ ]:
V.why_64(d)
print()
V.is_it_padding(d);

**The extra dimensions are empty — for now.** The embedding matrix is `64 x 49`, so its rank
is at most 49, and the singular values of 64,000 real tokens fall off a cliff at exactly index 49.
Every token lives in a 49-dimensional slice of the 64-d space.

They are not waste: `d_model` is the width of the **residual stream for the whole network**, and
every sub-layer *adds* to that stream, so the spare directions are scratch space the blocks write
into. (Three things constrain the choice of 64: it must divide by the head count — you could not
use 49 with 4 heads at all; GPU matmuls like multiples of 8/16/64; and it must be big enough that
`49 -> d` stays injective.)

**And it is a transformation, not padding:** no output equals any pixel, none is zero, and
changing a *single* pixel moves all 64 outputs. `W` is 64 rows of 49 weights — row *j* is a 7x7
filter, and output *j* answers "how much does this patch look like filter *j*?"

> This is not a teaching simplification. `timm` and the paper authors' own JAX code both write
> the same thing — `Conv2d(in_chans, embed_dim, kernel_size=patch_size, stride=patch_size)` then
> `flatten(2).transpose(1,2)` — only with `3 -> 768` at 16x16 instead of `1 -> 64` at 7x7. Their
> comment: *"We can merge s2d+emb into a single conv; it's the same."*

---
## 2 · What the model is

In [ ]:
display(Image(V.draw_architecture(d)))

Compare it with the CNN diagrams it imitates. **A CNN's slabs shrink** — 28 to 24 to 12 to
8 to 4 — because every conv and pool trades resolution for channels. **The ViT's slab is
`17 x 64` at the first block and `17 x 64` at the last.** No pooling, no stride, nothing shrinks.
A CNN builds a pyramid; a transformer keeps one fixed workspace and rewrites it in place.

Two things the figure marks that are easy to miss. Every tensor carries a leading **batch** axis
— training is 256 images at a time, 235 batches per epoch, 1,410 weight updates in all. And
**orange = learned**: 204,298 numbers, every one updated by the same `loss.backward()` +
`opt.step()`. `Q·Kᵀ`, the scaling, softmax, `@V` and the residual adds have **no parameters at
all**, which is why the layer's size is independent of sequence length.

If 256 images share a tensor, do they contaminate each other?

In [ ]:
V.batch_check(d);

**No** — image 0's logits are the same to `1e-07` whether it travels with 255 companions or
alone, because the attention tensor is 256 *separate* `17x17` matrices and nothing contracts the
batch axis. The one place the batch genuinely combines is the **loss**, which averages over all
256: one gradient, applied once.

Now follow one batch all the way through. Every shape below comes from a live tensor.

In [ ]:
V.trace_shapes(d);

Four things to notice.

**The image stops being an image on line 3.** `Conv2d` still returns a 4x4 grid; `flatten+transpose`
collapses it to a sequence. After that line the 2-D layout is gone.

**The `17x17` is the whole mechanism** — the only step where information crosses between
positions. LayerNorm, the MLP and every Linear work on one token at a time.

**`(256, 17, 64)` in, `(256, 17, 64)` out, four times.** That invariance is why blocks stack.

**Only `[CLS]` survives.** `t[:, 0]` keeps row 0 and discards 16 tokens — after four layers of
reading them.

### The 4 heads, and what Q, K, V look like

`HEADS = 4`, so each block has 4 heads and the model has 16. The heads **split** the width:
head 0 takes dims 0-15, head 1 takes 16-31, and so on. That is why `d_model` must divide by the
head count — `49/4 = 12.25`, and PyTorch refuses the reshape outright.

In [ ]:
display(Image(V.show_qkv(d)))

**Top row.** `x`, then `Q`, `K`, `V` — three re-expressions of the *same* 17 tokens, each
from its own learned matrix. `x` is visibly banded (rows 5, 6, 10, 13 are the inked patches);
`Q`/`K`/`V` lose that banding because each projection mixes all 64 dimensions.

**Bottom row.** The four `17x17` maps are visibly **different from one another** — head 0
concentrates on a couple of key columns, head 3 has a strong vertical stripe, head 2 is diffuse.
That difference is the entire argument for multi-head.

### `[CLS]` and `position`

Two learned things added at the same step, but in different ways — and the code settles which is
which:

```python
t = torch.cat([self.cls.expand(...), t], 1)   # 16 -> 17   CONCATENATED
t = t + self.pos                              # 17 -> 17   ADDED
```

`position` is a learned `17 x 64` matrix, literally added — **a per-slot bias**. `[CLS]` is a
learned 64-vector concatenated as a **17th token**; CLS is short for *classification*, the token
the classifier reads. It is not a bias: it creates a slot and gets its own q, k, v. Together
64 + 17x64 = **1,152**.

`[CLS]` is 64 ordinary floats, **not a label** — the digits 0-9 appear only as the 10 outputs of
`Linear(64, 10)`. And at the input it is byte-identical for every image. So where do its final
64 floats come from?

In [ ]:
V.trace_cls(d);

**They are accumulated.** `[CLS]` enters at `|v| = 0.05` with no image content and leaves at
`7.2`, because every sub-layer **adds** rather than replaces — the residual stream. Each of the
64 numbers is a learned constant plus 8 increments.

| | what it adds | can it see other tokens? |
|---|---|---|
| attention | a weighted sum of the 16 patch tokens' values | **yes** — the only step that can |
| MLP | a reshaping of this token alone | no |

So **attention supplies everything that came from the image.** Without it `[CLS]` would finish at
`0.19`, still identical for every image.

---
## 3 · How it learns

Five lines, once per batch:

```python
logits = model(x)                  # guess
loss   = cross_entropy(logits, y)  # how wrong, as ONE number
opt.zero_grad()                    # clear last batch's gradients
loss.backward()                    # d(loss)/d(every one of 204,298 parameters)
opt.step()                         # nudge each parameter downhill
```

In [ ]:
display(Image(V.show_training(d)))

**"6 epochs" is 1,410 updates** — 235 batches, six times over. The per-epoch printout samples
a curve doing far more than six things, and the number it prints is the **last batch's** loss,
one of 235 — which is why it jitters, sometimes upward. The blue moving average is the trend.

**Cross-entropy is `-log(probability given to the true class)`.** At init the model spreads
probability over 10 classes, so `-log(0.1) ≈ 2.3` — exactly where the curve starts. Accuracy and
loss ask different questions, which is why the orange curve keeps climbing after the blue one
flattens.

**Do the patch-embedding weights actually move?** Row *j* is a 7x7 filter, so we can look.

In [ ]:
display(Image(V.show_filters(d)))

**Barely.** `cosine(init, trained) ≈ 0.99`, and the third panel — on the same colour scale —
is almost blank. Yet accuracy went 10% to 95%.

On a task this small the patch embedding stays close to a **random projection** and the blocks
above learn to read it. That is a property of *this* setup, not a law: a large ViT's filters move
much further and end up as oriented edge detectors.

It is also the sharpest contrast with word2vec. Word embeddings are usually learned in a
**separate pretraining stage** with their own objective, then handed downstream as a finished
dictionary. A patch embedding has no such stage — pixels are already numeric — so it is learned
end to end from the classification loss, from the position furthest from that loss.

---
## 4 · The three measurements

Everything above was setup. These are the chapter's claims, measured.

### 4.1 It works, and the attention weights are readable

In [ ]:
m = V.exp_train(d)
display(Image("attention_maps.png"))

The bottom row is `[CLS]`'s attention over the 4x4 patch grid — **not uniform**. On a `1` the
mass runs up the stroke; on a `0` it sits around the ring. Attention weights are one of the few
interpretable things in a neural network, because they are a distribution over *inputs* rather
than over hidden features.

### 4.2 Position: the one thing attention cannot see

Two identical models, one with the positional encoding switched off, then **shuffle the patch
order at test time**. Predict the four numbers before running it.

In [ ]:
rows = V.exp_position(d);

The `+0.00` is the whole point. The model with no positional encoding is not merely *robust*
to the shuffle — it is **incapable of noticing it**, because all it ever received was a set of
vectors like the ones in Part 1, where the blank patches were byte-identical. That is permutation
equivariance, measured. The gap in the first column is what position buys.

### 4.3 Scale: what a prior is worth

Same budget, same gradient steps, growing training set. A CNN has locality **hard-coded**; the
transformer has to learn it.

In [ ]:
out = V.exp_scale(d);

**Read the gap column, not the accuracy columns.** The CNN's locality prior is free knowledge
handed over before training starts, and it is worth about **19 points at N = 1,000 and 3 points at
N = 60,000**.

No crossover here — the ViT never overtakes the CNN on MNIST, because all of MNIST sits inside the
small-data regime (the crossover in the ViT paper is past 100M images). What a laptop can show is
the **slope**: the prior paying off less and less. That slope is the argument.

---
## Things worth trying

- `V.PATCH = 4` (49 tokens) or `14` (4 tokens) — where does accuracy break, and is the cost
  compute or information?
- `V.LAYERS = 1` — how much does one round of mixing lose?
- `V.D = 32` or `128` — the width-vs-accuracy curve, 13 s per point.
- Shuffle the **pixels inside** each patch instead of the patches. Why does that hurt the
  no-position model, when shuffling patches did not?
- `PatchEmbed` is a `Conv2d` with `kernel_size == stride`. Make them differ so patches overlap —
  you have just reintroduced a locality prior. Does the N = 1,000 number move toward the CNN's?